# Chronoscope Raft chart demo

This notebook builds a temporary Chronoscope database from a gateway failover trace with many transactions and displays its Raft timelines. Run the cells from top to bottom. The source trace is not modified.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = "https://github.com/just-now/chronoscope.git"
BRANCH = "feat/chronoscope-event-relation"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content/chronoscope")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(repo_dir)],
            check=True,
        )
else:
    repo_dir = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "setup.py").exists() and (path / "chronoscope").is_dir()),
        None,
    )
    if repo_dir is None:
        raise RuntimeError("Start Jupyter from a Chronoscope checkout, or open this notebook in Colab.")

os.chdir(repo_dir)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--editable", str(repo_dir), "ipympl"],
    check=True,
)
print(f"Using Chronoscope from {repo_dir}")

In [ ]:
import importlib
importlib.invalidate_caches()

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

# Colab may import Matplotlib before this notebook installs ipympl. Refresh
# the two backend aliases if Matplotlib cached its entry points too early.
from matplotlib.backends import backend_registry
if not backend_registry.is_valid_backend("widget"):
    for alias in ("widget", "ipympl"):
        backend_registry._name_to_module[alias] = "module://ipympl.backend_nbagg"
        backend_registry._backend_to_gui_framework[alias] = "unknown"

get_ipython().run_line_magic("matplotlib", "widget")

## Build the demo database

The database lives in a temporary directory and is removed when the notebook kernel stops.

In [ ]:
import tempfile

from chronoscope import db, db_options
from chronoscope.parser import parser

demo_directory = tempfile.TemporaryDirectory()
demo_db = Path(demo_directory.name) / "raft_demo.db"
config = repo_dir / "test" / "raft_chronoscope.yaml"
trace = repo_dir / "examples" / "data" / "gateway_failover_many_transactions_trace.txt"

db.open(str(demo_db), db_options, create=True)
try:
    db.load(parser(str(config)), str(trace))
    db.mkidx()
    raft_ids = [
        row[0]
        for row in (
            db.state_machine
            .select(db.state_machine.id)
            .where(db.state_machine.type == "raft")
            .tuples()
        )
    ]
    if not raft_ids:
        raise RuntimeError("The trace contains no Raft state machines.")

    top_sm_id = (
        db.state_machine
        .select(db.state_machine.id)
        .order_by(db.state_machine.id)
        .scalar()
        - 1
    )
    with db.db.atomic():
        db.state_machine.create(id=top_sm_id, name="top", type="top")
        db.state_machine_relation.insert_many([
            {
                "from_sm_id": top_sm_id,
                "to_sm_id": raft_id,
                "relation": "top-to-raft",
            }
            for raft_id in raft_ids
        ]).execute()
finally:
    db.close()

print(f"Loaded {len(raft_ids)} Raft state machines; chart root: {top_sm_id:#x}")

## Display the Raft timelines

The toolbar supports pan, zoom, and saving the chart. To measure an interval, focus the chart and repeat `e` then click for each of its two endpoints. Press `d` to remove the latest marker.

In [ ]:
from chronoscope import chart

db.open(str(demo_db), db_options)
try:
    chart.plot(top_sm_id, figsize=(16, 28))
finally:
    db.close()

## Try another trace

Replace `config` and `trace` above with paths to files using the same Chronoscope formats, then rerun the last two code cells. In Colab, files can be uploaded through the Files panel.